# D2：倍率-容量衰减 —— Peukert 效应
# D2: Rate-Capacity Fade — The Peukert Effect

## 一、实验原理 | Principle

### 1.1 倍率效应：为什么大电流放不出全部容量？
同一块电池，0.5C 慢放能放出 5.00 Ah，3C 快放却只能放出 4.52 Ah——**快放"吃掉"了近 10% 的容量**。
这不是电池坏了，而是电化学动力学的必然结果。

*Rate effect: the same cell delivers 5.00 Ah at 0.5C but only 4.52 Ah at 3C. Fast discharge "eats" ~10% of the capacity — not a defect, but an inevitable result of electrochemical kinetics.*

### 1.2 Peukert 方程（1897）
德国科学家 Peukert 发现铅酸电池的放电时间与电流服从幂律：

$$t = \frac{C_p}{I^k}$$

其中 $t$ 为放电时间，$I$ 为放电电流，$C_p$ 为 Peukert 容量常数，$k \geq 1$ 为 **Peukert 指数**。
改写成放出容量 $Q = I \cdot t$ 的形式：

$$Q = \frac{C_p}{I^{k-1}}$$

取对数得直线关系：$\ln Q = \ln C_p - (k-1)\ln I$。
斜率的相反数就是 $(k-1)$——**斜率越陡，倍率效应越严重**。
锂离子电池的 $k$ 通常在 1.05–1.15 之间（铅酸电池可达 1.3）。

### 1.3 物理机制：三重极化
大电流下，端电压被三重极化压低，提前触及截止电压（2.5 V），
导致放电**提前终止**，深层 SOC 的容量放不出来：

$$V(I) = OCV(SOC) - \eta_{ohm}(I) - \eta_{act}(I) - \eta_{conc}(I, SOC)$$

- **欧姆极化** $\eta_{ohm} = I \cdot R$：与电流成正比，瞬间响应
- **活化极化** $\eta_{act}$：电荷转移需要过电位，Butler-Volmer 关系
- **浓差极化** $\eta_{conc}$：大电流下锂离子来不及扩散，表面浓度先耗尽

### 1.4 在"车型设计链"中的位置
```
D1 (单倍率仿真) → 【D2 (倍率-容量衰减)】 → D3 (OCV-SOC 标定) → C1 (FASTSim 整车)
```
D2 回答一个关键设计问题：**标称容量 5 Ah 的电池，在 2C/3C 大功率工况下到底能用多少？**
能量管理策略（E4）必须按倍率折减可用容量，否则续航预测会虚高。

## 二、实验目的 | Objectives
1. 理解倍率效应与 Peukert 方程的物理意义
2. 用 **PyBaMM 真实引擎**（SPM + Chen2020）跑 0.5C/1C/2C/3C 恒流放电
3. 拟合 Peukert 指数 $k$，量化所用电池的倍率敏感度
4. 将结果存入 `results/`，供 E4 能量管理策略调用



In [ ]:
import os
os.environ['PYBAMM_DISABLE_TELEMETRY'] = '1'  # suppress telemetry prompt on first run
import sys
sys.path.insert(0, '/home/jovyan/labs')
import numpy as np
import matplotlib.pyplot as plt
import pybamm
pybamm.set_logging_level('ERROR')
from common.vehicle_library import save_result, load_result
print('imports OK, pybamm', pybamm.__version__)


In [ ]:
# Run constant-current discharges at 0.5C/1C/2C/3C with the REAL PyBaMM engine
# (SPM electrochemical model + Chen2020 commercial NMC parameters).
# Chen2020 nominal capacity = 5.0 Ah, so 1C = 5.0 A.
# Each solve terminates on the 2.5 V cutoff event (or t_max); takes < 2 s wall time.
Q_NOM = 5.0  # Ah, nominal cell capacity

def run_discharge(current_a, t_max_s):
    """Run a real PyBaMM SPM discharge.
    Returns (discharged_capacity_Ah, end_voltage_V, discharge_time_s)."""
    model = pybamm.lithium_ion.SPM()
    param = pybamm.ParameterValues('Chen2020')
    param['Current function [A]'] = current_a  # PyBaMM 26.x: set current directly in Amperes
    sim = pybamm.Simulation(model, parameter_values=param)
    sim.solve([0, t_max_s])
    sol = sim.solution
    cap = sol['Discharge capacity [A.h]'].entries
    V = sol['Terminal voltage [V]'].entries
    t = sol['Time [s]'].entries
    return float(cap[-1]), float(V[-1]), float(t[-1])

# (current_A, sim_time_s, label)
rates = [(2.5, 7200, '0.5C'),
         (5.0, 3600, '1C'),
         (10.0, 1800, '2C'),
         (15.0, 1200, '3C')]
capacities, currents, times = [], [], []
for current_a, t_max, label in rates:
    cap, v_end, t_end = run_discharge(current_a, t_max)
    capacities.append(cap)
    currents.append(current_a)
    times.append(t_end)
    pct = cap / Q_NOM * 100
    print(f'{label} ({current_a:5.1f} A): {cap:.3f} Ah ({pct:.1f}% of nominal), '
          f'V_end={v_end:.3f} V, t={t_end:.0f} s')


In [ ]:
# Fit the Peukert equation: Q = Cp / I^(k-1)  =>  ln(Q) = ln(Cp) - (k-1)*ln(I)
# Linear regression of ln(Q) on ln(I); slope = -(k-1).
logI = np.log(np.array(currents))
logQ = np.log(np.array(capacities))
slope, intercept = np.polyfit(logI, logQ, 1)  # slope = -(k-1)
k_peukert = 1.0 - slope
Cp = float(np.exp(intercept))

print(f'Peukert exponent k = {k_peukert:.4f}')
print(f'Peukert capacity Cp = {Cp:.3f} Ah')
print(f'Capacity fade 0.5C -> 3C: {capacities[0]:.3f} -> {capacities[-1]:.3f} Ah '
      f'({(1 - capacities[-1]/capacities[0])*100:.1f}% loss)')

# --- Assertions: physics must hold ---
# 1. Capacity monotonically decreases with C-rate
for i in range(len(capacities) - 1):
    assert capacities[i] > capacities[i+1], \
        f'Capacity not monotonic: {capacities[i]:.3f} <= {capacities[i+1]:.3f}'
# 2. 1C capacity within 90% of nominal (Chen2020 ~4.96 Ah measured)
assert capacities[1] > 0.90 * Q_NOM, f'1C capacity {capacities[1]:.3f} Ah too low!'
# 3. Peukert exponent in physically reasonable range for Li-ion
assert 1.0 <= k_peukert <= 1.30, f'Peukert k={k_peukert:.3f} out of range!'
print('PASS: monotonic decrease, 1C > 90% nominal, k in [1.0, 1.30].')


In [ ]:
# Plot capacity vs C-rate with Peukert fit (English labels only; image has no CJK font)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: capacity vs C-rate (linear)
c_rates = [0.5, 1.0, 2.0, 3.0]
axes[0].plot(c_rates, capacities, 'o-', markersize=8, label='PyBaMM SPM')
I_fit = np.linspace(2.0, 16.0, 50)
Q_fit = Cp / I_fit**(k_peukert - 1)
axes[0].plot(I_fit / Q_NOM, Q_fit, '--', alpha=0.7, label=f'Peukert fit (k={k_peukert:.3f})')
axes[0].set_xlabel('C-rate')
axes[0].set_ylabel('Discharged capacity (Ah)')
axes[0].set_title('Rate-capacity fade (PyBaMM SPM, Chen2020)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: log-log Peukert linearization
axes[1].plot(logI, logQ, 'o', markersize=8, label='data')
axes[1].plot(logI, slope * logI + intercept, '-', alpha=0.7, label='linear fit')
axes[1].set_xlabel('ln(Current [A])')
axes[1].set_ylabel('ln(Capacity [Ah])')
axes[1].set_title(f'Peukert linearization (slope={slope:.4f})')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Publish this experiment's result for the downstream chain (E4 energy management uses it)
result_path = save_result('D2_rate_capacity', {
    'metrics': {
        'peukert_k': round(float(k_peukert), 4),
        'peukert_Cp_Ah': round(float(Cp), 3),
        'capacity_0p5C_Ah': round(float(capacities[0]), 3),
        'capacity_1C_Ah': round(float(capacities[1]), 3),
        'capacity_2C_Ah': round(float(capacities[2]), 3),
        'capacity_3C_Ah': round(float(capacities[3]), 3),
        'fade_0p5C_to_3C_pct': round((1 - capacities[-1]/capacities[0])*100, 2),
    },
    'params': {'model': 'SPM', 'parameter_set': 'Chen2020',
               'nominal_capacity_Ah': Q_NOM, 'cutoff_voltage_V': 2.5},
})
print(f'Result saved: {result_path}')

# Downstream demo: E4 would do exactly this
r = load_result('D2_rate_capacity')
print('Downstream (E4) sees Peukert k =', r['metrics']['peukert_k'])


## 三、学生任务 | Your Tasks
1. **运行** 四种倍率放电仿真，记录每种倍率的放出容量。为什么 0.5C 没有触发 2.5 V 截止？（提示：看 `t` 是否跑满 `t_max`）
2. **拟合**：用 `np.polyfit` 自己算一遍 Peukert 指数，验证 notebook 给出的 $k$ 值。
3. **对比**：把 D2 的 Peukert $k$ 与文献值对比——锂离子电池 $k$ 通常 1.05–1.15，SPM 算出的偏小还是偏大？为什么？（提示：SPM 忽略了电解液浓差极化）
4. **思考题**：如果把截止电压从 2.5 V 放宽到 2.0 V，3C 能多放出多少容量？这对电池寿命有什么代价？
5. **🏆 挑战**：跑 5C（25 A）放电，PyBaMM 是否还能收敛？记录容量并外推 Peukert 曲线，评估外推误差。

## 四、验证内容 | Verification
- [ ] 四种倍率 PyBaMM 仿真全部收敛（无求解器报错）
- [ ] 容量随倍率单调递减：0.5C > 1C > 2C > 3C
- [ ] 1C 容量 > 90% 标称（> 4.50 Ah）
- [ ] Peukert 指数 $k$ 在合理范围 [1.0, 1.30]
- [ ] 0.5C→3C 容量衰减约 8–12%（真机实测量级）
- [ ] `save_result` 成功保存结果 JSON，下游可加载

### 常见报错说明 | Troubleshooting
| 报错信息 | 中文解释 | 解决方法 |
|---------|---------|---------|
| `KeyError: 'Typical current [A]'` | PyBaMM 新版删掉了该参数名，旧教程代码失效 | 直接用安培数设电流：`param['Current function [A]'] = 5.0` |
| 首次运行卡住问 telemetry | PyBaMM 询问是否发送匿名使用统计 | 代码开头加 `os.environ['PYBAMM_DISABLE_TELEMETRY'] = '1'` |
| `Solver failed` 或 `Step failed` | 高倍率下求解器不收敛 | SPM 通常没问题；检查电流是否超出合理范围（如 > 5C） |
| 容量不随倍率递减 | 仿真时间 `t_max` 太短，低倍率没放完 | 0.5C 的 `t_max` 至少 7200 s（2 小时） |
| Peukert $k < 1.0$ | 拟合数据有问题 | 检查是否误用了未截止的 0.5C 时间数据做拟合；本实验用容量 $Q$ 拟合，不受此影响 |
| `ValueError: xp must be increasing` | `np.interp` 的 x 数组非递增 | 本实验不需要 interp；如需用，先 `np.argsort` 排序 |

